# Time-Distance Batch Runner

Use this notebook to batch the existing `time_distance.py` workflow in either `paired_cubes` or `single_cube` mode. Each batch cell builds run configurations, executes the pipeline, and can execute `td_analysis.ipynb` for every generated run.

In [1]:
from __future__ import annotations

import copy
import importlib.util
import itertools
import pprint
import re
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Iterable

import numpy as np


ANALYSIS_INPUT_CELL_MARKER = "# Load the shared configuration file and define the plot requests"

_ITER_RUN_ARG_DEFAULTS = {
    "source_type": None,
    "files": None,
    "observable": None,
    "h1": None,
    "h2": None,
    "height_pair": None,
    "file_1": None,
    "file_2": None,
    "file_pair": None,
    "delta_z_km": None,
    "p_dx_Mm": None,
    "dt": None,
}


def load_local_module(module_name: str, file_path: str | Path):
    '''
    Purpose
    -------
    Load local module.
    
    Inputs
    ------
    module_name: str
        Module name to load.
    
    file_path: str | Path
        File path used by this helper.
    
    Outputs
    -------
    loaded_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    file_path = Path(file_path).expanduser().resolve()
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    # Fail early when importlib cannot create a usable module specification.
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module {module_name!r} from {file_path}.")

    module = importlib.util.module_from_spec(spec)
    # Execute the discovered loader so the imported module is ready for use.
    spec.loader.exec_module(module)
    # Return the imported module object for reuse in the batch helpers.
    return module



def resolve_td_batch_module_dir(config_file: str | Path | None = None) -> Path:
    '''
    Purpose
    -------
    Resolve time-distance batch module dir.
    
    Inputs
    ------
    config_file: str | Path | None
        Path to the configuration file.
    
    Outputs
    -------
    resolved_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Collect candidate directories that might contain the batch notebook and pipeline.
    candidates: list[Path] = []

    # Prefer candidate directories derived from the explicit config file when one was provided.
    if config_file not in ['', None]:
        resolved_config = Path(config_file).expanduser().resolve()
        candidates.extend([
            resolved_config.parent,
            resolved_config.parent / 'Code' / 'Time-Distance',
        ])

    candidates.extend([
        Path.cwd().resolve(),
        (Path.cwd() / 'Code' / 'Time-Distance').resolve(),
    ])

    seen: set[Path] = set()
    # Check each candidate once and return the first valid Time-Distance directory.
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)

        if (candidate / 'time_distance.py').exists() and (candidate / 'td_analysis.ipynb').exists():
            # Return the assembled helper result.
            return candidate

    raise FileNotFoundError(
        'Could not resolve the Code/Time-Distance directory from the current working directory. '
        'Run td_batch.ipynb from the project root or from Code/Time-Distance, or set CONFIG_FILE explicitly.'
    )


def resolve_time_distance_environment(config_file: str | Path | None = None) -> dict[str, Any]:
    '''
    Purpose
    -------
    Resolve time distance environment.
    
    Inputs
    ------
    config_file: str | Path | None
        Path to the configuration file.
    
    Outputs
    -------
    resolved_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the shared batch workspace before deriving any runtime paths.
    module_dir = resolve_td_batch_module_dir(config_file = config_file)
    # Derive the local time-distance script path from the resolved module directory.
    time_distance_file = module_dir / "time_distance.py"
    # Derive the local analysis notebook path from the resolved module directory.
    analysis_notebook = module_dir / "td_analysis.ipynb"
    # Load the local time-distance module directly from the project tree.
    time_distance_module = load_local_module("time_distance_batch_runtime", time_distance_file)
    # Let time_distance.py resolve the active configuration file path.
    resolved_config_file = time_distance_module.resolve_config_file(config_file)

    # Return the assembled helper result.
    return {
        "module_dir": module_dir,
        "time_distance_file": time_distance_file,
        "analysis_notebook": analysis_notebook.resolve(),
        "config_file": resolved_config_file,
        "time_distance_module": time_distance_module,
    }


def load_mode_base_config(
    source_type: str,
    config_file: str | Path | None = None,
) -> tuple[Path, Any, dict[str, Any]]:
    '''
    Purpose
    -------
    Load mode base config.
    
    Inputs
    ------
    source_type: str
        Source-type label used by this helper.
    
    config_file: str | Path | None
        Path to the configuration file.
    
    Outputs
    -------
    loaded_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Load the shared runtime environment and active config for this mode.
    environment = resolve_time_distance_environment(config_file = config_file)
    # Let time_distance.py resolve the active configuration file path.
    resolved_config_file = environment["config_file"]
    time_distance_module = environment["time_distance_module"]
    # Load the Python config module that defines the batch defaults.
    config_module = time_distance_module.load_config_module(resolved_config_file)
    # Normalize the requested source type before selecting its default inputs.
    normalized_source_type = time_distance_module._normalize_source_type(source_type)

    # Prefer the config helper when the module exposes one.
    if hasattr(config_module, "get_config"):
        # Format paired-cube runs using the two file names.
        if normalized_source_type == "paired_cubes":
            # Copy the mode-specific default inputs so the config helper can be called safely.
            default_inputs = copy.deepcopy(getattr(config_module, "paired_cubes_inputs", {}))
        # Format single-cube runs with the observable and sampled heights.
        elif normalized_source_type == "single_cube":
            default_inputs = copy.deepcopy(getattr(config_module, "single_cube_inputs", {}))
        else:
            raise ValueError(f"Unsupported source_type: {source_type}")

        # Build the base config through the config helper so mode defaults are applied consistently.
        base_config = config_module.get_config(
            source_type = normalized_source_type,
            **default_inputs,
        )
    else:
        base_config = copy.deepcopy(config_module.config)
        base_config["data"]["source_type"] = normalized_source_type

    # Return the assembled helper result for downstream batch logic.
    return resolved_config_file, time_distance_module, copy.deepcopy(base_config)


def normalize_cube_paths(cube_paths: Iterable[str | Path]) -> list[str]:
    '''
    Purpose
    -------
    Normalize cube paths.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Work with normalized paths so duplicate or relative inputs collapse to one entry.
    normalized_paths: list[str] = []
    seen: set[str] = set()

    # Inspect each cube path in turn and collect the derived metadata.
    for cube_path in cube_paths:
        # Ignore empty cube-path placeholders so optional notebook lists remain easy to edit.
        if cube_path in ["", None]:
            continue

        resolved_path = str(Path(cube_path).expanduser().resolve())

        # Skip duplicate paths after normalization so each run is generated only once.
        if resolved_path in seen:
            continue

        seen.add(resolved_path)
        normalized_paths.append(resolved_path)

    # Return the assembled helper result for downstream batch logic.
    return normalized_paths


def generate_unique_unordered_pairs(cube_paths: Iterable[str | Path]) -> list[tuple[str, str]]:
    '''
    Purpose
    -------
    Generate unique unordered pairs.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    generated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)
    # Return the assembled helper result for downstream batch logic.
    return [(file_1, file_2) for file_1, file_2 in itertools.combinations(normalized_paths, 2)]


def generate_height_index_pairs(number_of_heights: int) -> list[tuple[int, int]]:
    '''
    Purpose
    -------
    Generate height index pairs.
    
    Inputs
    ------
    number_of_heights: int
        Number of heights used by this helper.
    
    Outputs
    -------
    generated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    number_of_heights = int(number_of_heights)

    # Reject negative height counts before building combinations.
    if number_of_heights < 0:
        raise ValueError("number_of_heights must be non-negative.")

    # Return the assembled helper result for downstream batch logic.
    return [(h1, h2) for h1, h2 in itertools.combinations(range(number_of_heights), 2)]


def parse_single_cube_field_strength_case(cube_path: str | Path) -> dict[str, Any]:
    '''
    Purpose
    -------
    Parse single cube field-strength case.
    
    Inputs
    ------
    cube_path: str | Path
        Cube path used by this helper.
    
    Outputs
    -------
    parsed_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the cube path once so the encoded field-strength case can be parsed reliably.
    resolved_path = str(Path(cube_path).expanduser().resolve())
    # Inspect the path components because these comparison helpers infer metadata from folder names.
    path = Path(resolved_path)
    # Initialize the parsed case metadata before scanning the path.
    component = ""
    strength_token = ""

    # Scan the path components to infer the encoded comparison metadata.
    for part in path.parts:
        lower_part = part.lower()

        if lower_part in ["hx", "vx", "z0"]:
            # Initialize the parsed case metadata before scanning the path.
            component = lower_part

        if re.fullmatch(r"\d+(?:[._]\d+)?g", lower_part):
            # Initialize the parsed case metadata before scanning the path.
            strength_token = lower_part

    # Fall back to the encoded path name when the folder scan is not enough.
    if component == "" or strength_token == "":
        match = re.search(
            r"(hx|vx|z0)[_\-/](\d+(?:[._]\d+)?g)",
            resolved_path,
            flags = re.IGNORECASE,
        )

        if match is not None:
            # Initialize the parsed case metadata before scanning the path.
            component = match.group(1).lower()
            strength_token = match.group(2).lower()

    # Limit the field-strength comparison to the supported magnetic geometries.
    if component not in ["hx", "vx"]:
        raise ValueError(
            "Field-strength comparison requires cube paths that encode either 'hx' or 'vx' geometry. "
            f"Could not infer a supported geometry from {resolved_path}."
        )

    # Require an encoded field strength for this comparison mode.
    if strength_token == "":
        raise ValueError(
            "Field-strength comparison requires cube paths that encode a field-strength folder such as "
            f"'10G'. Could not infer the field strength from {resolved_path}."
        )

    # Convert the encoded field strength into both a numeric value and a stable display label.
    field_strength_G = float(strength_token[:-1].replace("_", "."))
    strength_label_value = (
        f"{int(round(field_strength_G))}"
        if np.isclose(field_strength_G, round(field_strength_G))
        else f"{field_strength_G:g}"
    )

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_path": resolved_path,
        "component": component,
        "geometry": "horizontal" if component == "hx" else "vertical",
        "field_strength_G": field_strength_G,
        "field_strength_token": strength_token,
        "field_strength_label": f"{strength_label_value} G",
    }


def organize_single_cube_field_strength_cases(cube_paths: Iterable[str | Path]) -> dict[str, Any]:
    '''
    Purpose
    -------
    Organize single cube field-strength cases.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    organized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("Field-strength comparison requires at least one single_cube path.")

    # Group the parsed field-strength cases by geometry for predictable plotting order.
    cases_by_geometry: dict[str, list[dict[str, Any]]] = {
        "horizontal": [],
        "vertical": [],
    }
    # Track geometry-strength combinations so duplicate cases can be rejected early.
    seen_case_keys: dict[tuple[str, str], str] = {}

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths:
        case = parse_single_cube_field_strength_case(cube_path)
        case_key = (case["geometry"], case["field_strength_token"])

        # Reject duplicate geometry-strength cases so each comparison panel is unique.
        if case_key in seen_case_keys:
            raise ValueError(
                "Duplicate field-strength comparison case detected for "
                f"{case['geometry']} {case['field_strength_label']}: "
                f"{seen_case_keys[case_key]} and {case['cube_path']}"
            )

        # Track geometry-strength combinations so duplicate cases can be rejected early.
        seen_case_keys[case_key] = case["cube_path"]
        cases_by_geometry[case["geometry"]].append(case)

    # Sort each geometry bucket in place so the plotting order stays stable.
    for geometry_cases in cases_by_geometry.values():
        geometry_cases.sort(key = lambda item: (item["field_strength_G"], item["cube_path"]))

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_paths": normalized_paths,
        "cases_by_geometry": cases_by_geometry,
        "field_strengths_by_geometry": {
            geometry: [case["field_strength_G"] for case in geometry_cases]
            for geometry, geometry_cases in cases_by_geometry.items()
        },
        "ordered_cases": cases_by_geometry["horizontal"] + cases_by_geometry["vertical"],
    }


def normalize_gaussian_filter_param_set(
    filter_params: dict[str, Any],
    *,
    reference_config: dict[str, Any] | None = None,
    filter_index: int | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Normalize Gaussian-filter param set.
    
    Inputs
    ------
    filter_params: dict[str, Any]
        Gaussian-filter parameters used by this helper.
    
    reference_config: dict[str, Any] | None
        Reference config used by this helper.
    
    filter_index: int | None
        Filter index used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if not isinstance(filter_params, dict):
        raise TypeError("Each Gaussian filter parameter set must be provided as a dictionary.")

    raw_gaussian = filter_params.get("gaussian", filter_params)
    if not isinstance(raw_gaussian, dict):
        raise TypeError("Gaussian filter parameters must be stored in a dictionary.")

    # Start from the reference Gaussian defaults when they are available.
    gaussian_defaults = {}
    # Inherit any unspecified Gaussian settings from the reference config when available.
    if reference_config is not None:
        # Start from the reference Gaussian defaults when they are available.
        gaussian_defaults = copy.deepcopy(reference_config.get("filtering", {}).get("gaussian", {}))

    # Accumulate the runtime Gaussian settings without mutating the reference config.
    gaussian_config = copy.deepcopy(gaussian_defaults)
    # Copy each runtime Gaussian parameter except the label-style metadata fields.
    for key, value in raw_gaussian.items():
        # Skip display metadata keys because they do not belong in the runtime filter config.
        if key in ["label", "name", "slug"]:
            continue
        # Accumulate the runtime Gaussian settings without mutating the reference config.
        gaussian_config[key] = value

    # Accumulate the runtime Gaussian settings without mutating the reference config.
    gaussian_config["enabled"] = bool(gaussian_config.get("enabled", True))

    # Require the four shape parameters that define the Gaussian filter.
    required_keys = ["central_k", "width_k", "central_f", "width_f"]
    missing_keys = [key for key in required_keys if gaussian_config.get(key, None) in ["", None]]
    if len(missing_keys) > 0:
        raise ValueError(
            "Each Gaussian filter parameter set must define "
            f"{', '.join(missing_keys)}."
        )

    # Return the assembled helper result for downstream batch logic.
    return {
        "filter_index": None if filter_index is None else int(filter_index),
        "label": str(filter_params.get("label", filter_params.get("name", ""))),
        "gaussian": gaussian_config,
    }


def normalize_gaussian_filter_param_list(
    filter_params_list: Iterable[dict[str, Any]],
    *,
    reference_config: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    '''
    Purpose
    -------
    Normalize Gaussian-filter param list.
    
    Inputs
    ------
    filter_params_list: Iterable[dict[str, Any]]
        Gaussian-filter parameter sets used by this helper.
    
    reference_config: dict[str, Any] | None
        Reference config used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Collect the normalized filter specifications in the requested order.
    normalized_filter_params: list[dict[str, Any]] = []

    # Normalize each requested Gaussian filter in the order it was provided.
    for filter_index, filter_params in enumerate(filter_params_list):
        normalized_filter_params.append(
            normalize_gaussian_filter_param_set(
                filter_params,
                reference_config = reference_config,
                filter_index = filter_index,
            )
        )

    # Require at least one Gaussian filter specification before building the comparison plan.
    if len(normalized_filter_params) == 0:
        raise ValueError("Gaussian-filter comparison requires at least one Gaussian filter parameter set.")

    # Return the assembled helper result for downstream batch logic.
    return normalized_filter_params


def apply_gaussian_filter_params_to_config(
    base_config: dict[str, Any],
    filter_params: dict[str, Any],
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Apply Gaussian-filter params to config.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    filter_params: dict[str, Any]
        Gaussian-filter parameters used by this helper.
    
    Outputs
    -------
    updated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Work on a deep copy so shared base-config state is not mutated in place.
    runtime_config = copy.deepcopy(base_config)
    # Normalize the Gaussian filter settings before injecting them into the runtime config.
    normalized_filter_params = normalize_gaussian_filter_param_set(
        filter_params,
        reference_config = runtime_config,
    )

    # Access the filtering section that will be updated with the Gaussian settings.
    filtering = runtime_config.setdefault("filtering", {})
    filter_sequence = list(filtering.get("filter_sequence", []))
    # Append the Gaussian stage when it is not already present in the filter order.
    if "gaussian" not in filter_sequence:
        filter_sequence.append("gaussian")
    filtering["filter_sequence"] = filter_sequence
    filtering["enabled"] = True
    filtering["gaussian"] = copy.deepcopy(normalized_filter_params["gaussian"])

    # Return the assembled helper result.
    return runtime_config


def parse_single_cube_gaussian_filter_comparison_case(cube_path: str | Path) -> dict[str, Any]:
    '''
    Purpose
    -------
    Parse single cube Gaussian-filter comparison case.
    
    Inputs
    ------
    cube_path: str | Path
        Cube path used by this helper.
    
    Outputs
    -------
    parsed_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the cube path once so the encoded Gaussian-comparison case can be parsed reliably.
    resolved_path = str(Path(cube_path).expanduser().resolve())
    # Inspect the path components because these comparison helpers infer metadata from folder names.
    path = Path(resolved_path)
    # Initialize the parsed case metadata before scanning the path.
    component = ""
    strength_token = ""

    # Scan the path components to infer the encoded comparison metadata.
    for part in path.parts:
        lower_part = part.lower()

        if lower_part in ["hx", "vx", "z0"]:
            # Initialize the parsed case metadata before scanning the path.
            component = lower_part

        if re.fullmatch(r"\d+(?:[._]\d+)?g", lower_part):
            # Initialize the parsed case metadata before scanning the path.
            strength_token = lower_part

    # Fall back to the encoded path name when the folder scan is not enough.
    if component == "" or strength_token == "":
        match = re.search(
            r"(hx|vx|z0)[_\-/](\d+(?:[._]\d+)?g)",
            resolved_path,
            flags = re.IGNORECASE,
        )

        if match is not None:
            # Initialize the parsed case metadata before scanning the path.
            component = match.group(1).lower()
            strength_token = match.group(2).lower()

    # Treat the 0G case as the non-magnetic reference even when the component folder is omitted.
    if component == "" and strength_token == "0g":
        # Initialize the parsed case metadata before scanning the path.
        component = "z0"

    # Fall back to the encoded path name when the folder scan is not enough.
    if component == "" or strength_token == "":
        raise ValueError(
            "Gaussian-filter comparison requires cube paths that encode a supported geometry folder "
            "(hx, vx, or z0) and a field-strength folder such as '10G'. "
            f"Could not infer those values from {resolved_path}."
        )

    # Convert the encoded field strength into both a numeric value and a stable display label.
    field_strength_G = float(strength_token[:-1].replace("_", "."))
    strength_label_value = (
        f"{int(round(field_strength_G))}"
        if np.isclose(field_strength_G, round(field_strength_G))
        else f"{field_strength_G:g}"
    )

    # Map the non-magnetic reference into the canonical comparison label and ordering key.
    if component == "z0" or np.isclose(field_strength_G, 0.0, rtol = 1.0e-9, atol = 1.0e-9):
        case_key = "0g"
        comparison_label = "0G"
        group_name = "zero"
    elif component == "hx":
        # Restrict the magnetic comparison cases to the supported 10G, 50G, and 100G strengths.
        if not any(np.isclose(field_strength_G, value, rtol = 1.0e-9, atol = 1.0e-9) for value in [10.0, 50.0, 100.0]):
            raise ValueError(
                "Gaussian-filter comparison supports only the horizontal 10G, 50G, and 100G cases. "
                f"Received {resolved_path}."
            )

        case_key = f"h{strength_label_value.lower()}g"
        comparison_label = f"h{strength_label_value}G"
        group_name = "horizontal"
    elif component == "vx":
        # Restrict the magnetic comparison cases to the supported 10G, 50G, and 100G strengths.
        if not any(np.isclose(field_strength_G, value, rtol = 1.0e-9, atol = 1.0e-9) for value in [10.0, 50.0, 100.0]):
            raise ValueError(
                "Gaussian-filter comparison supports only the vertical 10G, 50G, and 100G cases. "
                f"Received {resolved_path}."
            )

        case_key = f"v{strength_label_value.lower()}g"
        comparison_label = f"v{strength_label_value}G"
        group_name = "vertical"
    else:
        raise ValueError(
            "Gaussian-filter comparison requires hx, vx, or z0 simulation paths. "
            f"Could not infer a supported case from {resolved_path}."
        )

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_path": resolved_path,
        "component": component,
        "group_name": group_name,
        "field_strength_G": field_strength_G,
        "field_strength_token": strength_token,
        "field_strength_label": f"{strength_label_value} G",
        "case_key": case_key,
        "comparison_label": comparison_label,
    }


def organize_single_cube_gaussian_filter_comparison_cases(cube_paths: Iterable[str | Path]) -> dict[str, Any]:
    '''
    Purpose
    -------
    Organize single cube Gaussian-filter comparison cases.
    
    Inputs
    ------
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    Outputs
    -------
    organized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)
    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("Gaussian-filter comparison requires at least one single_cube path.")

    # Keep the required case order explicit because the comparison layout depends on it.
    required_order = ["0g", "h10g", "h50g", "h100g", "v10g", "v50g", "v100g"]
    # Map each parsed case to its canonical comparison key.
    case_lookup: dict[str, dict[str, Any]] = {}

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths:
        case = parse_single_cube_gaussian_filter_comparison_case(cube_path)

        if case["case_key"] in case_lookup:
            raise ValueError(
                "Duplicate Gaussian-filter comparison case detected for "
                f"{case['comparison_label']}: {case_lookup[case['case_key']]['cube_path']} and {case['cube_path']}"
            )

        # Map each parsed case to its canonical comparison key.
        case_lookup[case["case_key"]] = case

    missing_cases = [case_key.upper() for case_key in required_order if case_key not in case_lookup]
    if len(missing_cases) > 0:
        missing_labels = [case_lookup_key.replace("H", "h").replace("V", "v") for case_lookup_key in missing_cases]
        raise ValueError(
            "Gaussian-filter comparison requires the seven simulation cases "
            "0G, h10G, h50G, h100G, v10G, v50G, and v100G. "
            f"Missing cases: {', '.join(missing_labels)}."
        )

    # Materialize the cases in the canonical plotting order expected by the notebook.
    ordered_cases = [case_lookup[case_key] for case_key in required_order]

    # Return the assembled helper result for downstream batch logic.
    return {
        "cube_paths": [case["cube_path"] for case in ordered_cases],
        "case_lookup": case_lookup,
        "ordered_cases": ordered_cases,
        "ordered_labels": [case["comparison_label"] for case in ordered_cases],
    }


def infer_shared_single_cube_height_grid_km(
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    rtol: float = 1.0e-9,
    atol: float = 1.0e-6,
) -> list[float]:
    '''
    Purpose
    -------
    Infer shared single cube height grid km.
    
    Inputs
    ------
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    rtol: float
        Rtol used by this helper.
    
    atol: float
        Atol used by this helper.
    
    Outputs
    -------
    inferred_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("Need at least one single_cube path to infer a shared height grid.")

    # Use the first cube as the reference height grid for all subsequent comparisons.
    reference_path = normalized_paths[0]
    # Read the reference height coordinates from the first normalized cube.
    reference_heights = np.asarray(
        time_distance_module.infer_netcdf_height_coordinates_km(reference_path),
        dtype = np.float64,
    )

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths[1:]:
        # Read the comparison cube height grid before checking it against the reference.
        comparison_heights = np.asarray(
            time_distance_module.infer_netcdf_height_coordinates_km(cube_path),
            dtype = np.float64,
        )

        # Require every cube in the comparison set to share the same sampled height grid.
        if comparison_heights.shape != reference_heights.shape or not np.allclose(
            comparison_heights,
            reference_heights,
            rtol = float(rtol),
            atol = float(atol),
            equal_nan = True,
        ):
            raise ValueError(
                "All single_cube comparison paths must share the same height coordinate grid. "
                f"{reference_path} and {cube_path} do not match."
            )

    # Return the assembled helper result for downstream batch logic.
    return [float(value) for value in reference_heights]


def generate_shared_single_cube_height_index_pairs(
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    rtol: float = 1.0e-9,
    atol: float = 1.0e-6,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Generate shared single cube height index pairs.
    
    Inputs
    ------
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    rtol: float
        Rtol used by this helper.
    
    atol: float
        Atol used by this helper.
    
    Outputs
    -------
    generated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Read the shared height grid before enumerating the valid height pairs.
    height_values_km = infer_shared_single_cube_height_grid_km(
        time_distance_module,
        cube_paths,
        rtol = rtol,
        atol = atol,
    )

    # Return the assembled helper result for downstream batch logic.
    return {
        "height_values_km": height_values_km,
        "height_pairs": generate_height_index_pairs(len(height_values_km)),
    }


def _make_iter_run_args(**overrides: Any) -> SimpleNamespace:
    '''
    Purpose
    -------
    Create iter run args.
    
    Inputs
    ------
    overrides: Any
        Overrides used by this helper.
    
    Outputs
    -------
    created_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Start from the canonical iter_run_configs defaults before applying overrides.
    arguments = copy.deepcopy(_ITER_RUN_ARG_DEFAULTS)
    arguments.update(overrides)
    # Return the assembled helper result.
    return SimpleNamespace(**arguments)


def build_paired_cubes_batch_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    delta_z_km: float | None = None,
    p_dx_Mm: float | None = None,
    dt: float | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build paired cubes batch plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    delta_z_km: float | None
        Delta z km used by this helper.
    
    p_dx_Mm: float | None
        P dx Mm used by this helper.
    
    dt: float | None
        Dt used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least two unique paths before building paired-cube runs.
    if len(normalized_paths) < 2:
        raise ValueError("paired_cubes batch mode requires at least two unique cube paths.")

    # Build the unique unordered file pairs that feed the paired-cube batch.
    file_pairs = generate_unique_unordered_pairs(normalized_paths)
    # Build the iter_run_configs argument bundle expected by time_distance.py.
    run_args = _make_iter_run_args(
        source_type = "paired_cubes",
        file_pair = file_pairs,
        delta_z_km = delta_z_km,
        p_dx_Mm = p_dx_Mm,
        dt = dt,
    )
    # Expand the batch plan into the concrete runtime configs that will be executed.
    run_configs = time_distance_module.iter_run_configs(copy.deepcopy(base_config), run_args)

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "paired_cubes",
        "cube_paths": normalized_paths,
        "file_pairs": file_pairs,
        "v1_list": [file_1 for file_1, _ in file_pairs],
        "v2_list": [file_2 for _, file_2 in file_pairs],
        "run_configs": run_configs,
    }


def build_single_cube_batch_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube batch plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize and deduplicate the requested cube paths before building the plan.
    normalized_paths = normalize_cube_paths(cube_paths)

    # Require at least one normalized cube path before continuing.
    if len(normalized_paths) == 0:
        raise ValueError("single_cube batch mode requires at least one cube path.")

    # Initialize the run-plan containers that will be filled below.
    run_configs: list[dict[str, Any]] = []
    height_pairs_by_cube: dict[str, list[tuple[int, int]]] = {}
    height_values_km_by_cube: dict[str, list[float]] = {}
    height_pair_rows: list[dict[str, Any]] = []

    # Resolve the optional model-atmosphere path once so every generated config uses the same file.
    resolved_model_atmosphere = (
        str(Path(model_atmosphere_path).expanduser().resolve())
        if model_atmosphere_path not in ["", None]
        else None
    )

    # Iterate through the normalized cube paths so every case is handled once.
    for cube_path in normalized_paths:
        # Read the available height coordinates for the current cube or shared grid.
        height_values_km = [
            float(value)
            for value in time_distance_module.infer_netcdf_height_coordinates_km(cube_path)
        ]
        # Enumerate every valid h1 < h2 pair for the sampled height grid.
        height_pairs = generate_height_index_pairs(len(height_values_km))
        height_pairs_by_cube[cube_path] = height_pairs
        height_values_km_by_cube[cube_path] = height_values_km

        # Iterate over every valid shared height pair.
        for h1_value, h2_value in height_pairs:
            height_pair_rows.append(
                {
                    "file": cube_path,
                    "observable": observable if observable not in ["", None] else base_config["data"]["single_cube"]["observable"],
                    "h1": int(h1_value),
                    "h2": int(h2_value),
                    "h1_km": float(height_values_km[h1_value]),
                    "h2_km": float(height_values_km[h2_value]),
                }
            )

        # Skip cubes that do not provide any valid h1 < h2 combinations.
        if len(height_pairs) == 0:
            continue

        cube_base_config = copy.deepcopy(base_config)
        cube_base_config["data"]["source_type"] = "single_cube"
        cube_base_config["data"]["single_cube"]["file"] = cube_path

        # Apply the observable override only when the caller provided one.
        if observable not in ["", None]:
            cube_base_config["data"]["single_cube"]["observable"] = str(observable)

        # Apply the optional model-atmosphere path when one was supplied.
        if resolved_model_atmosphere is not None:
            cube_base_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

        # Build the iter_run_configs argument bundle expected by time_distance.py.
        run_args = _make_iter_run_args(
            source_type = "single_cube",
            files = [cube_path],
            observable = observable,
            height_pair = height_pairs,
        )
        run_configs.extend(time_distance_module.iter_run_configs(cube_base_config, run_args))

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "single_cube",
        "cube_paths": normalized_paths,
        "height_pairs_by_cube": height_pairs_by_cube,
        "height_values_km_by_cube": height_values_km_by_cube,
        "height_pair_rows": height_pair_rows,
        "run_configs": run_configs,
    }


def build_single_cube_field_strength_comparison_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube field-strength comparison plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Organize the supplied cube paths into the canonical case ordering for this comparison mode.
    organized_cases = organize_single_cube_field_strength_cases(cube_paths)
    # Work with normalized paths so duplicate or relative inputs collapse to one entry.
    normalized_paths = organized_cases["cube_paths"]
    # Infer the shared height grid once so every comparison uses the same height pairs.
    shared_heights = generate_shared_single_cube_height_index_pairs(
        time_distance_module,
        normalized_paths,
    )

    # Require at least one shared height pair before building comparison requests.
    if len(shared_heights["height_pairs"]) == 0:
        raise ValueError(
            "Field-strength comparison requires at least two shared heights so h1 < h2 pairs exist."
        )

    # Resolve the optional model-atmosphere path once so every generated config uses the same file.
    resolved_model_atmosphere = (
        str(Path(model_atmosphere_path).expanduser().resolve())
        if model_atmosphere_path not in ["", None]
        else None
    )

    # Resolve the observable once so every run and plot request stays synchronized.
    default_observable = base_config.get("data", {}).get("single_cube", {}).get("observable", "")
    comparison_observable = str(observable) if observable not in ["", None] else str(default_observable)
    # Use one representative cube path because the plotting helper will receive the full case list separately.
    representative_cube_path = organized_cases["ordered_cases"][0]["cube_path"]
    # Accumulate the direct plot requests that td_analysis.ipynb will execute later.
    comparison_runs: list[dict[str, Any]] = []

    # Iterate over every valid shared height pair.
    for h1_value, h2_value in shared_heights["height_pairs"]:
        # Work on a deep copy so shared base-config state is not mutated in place.
        runtime_config = copy.deepcopy(base_config)
        runtime_config["data"]["source_type"] = "single_cube"
        runtime_config["data"].setdefault("single_cube", {})
        runtime_config["data"]["single_cube"]["file"] = representative_cube_path
        runtime_config["data"]["single_cube"]["observable"] = comparison_observable
        runtime_config["data"]["single_cube"]["h1"] = int(h1_value)
        runtime_config["data"]["single_cube"]["h2"] = int(h2_value)

        # Apply the optional model-atmosphere path when one was supplied.
        if resolved_model_atmosphere is not None:
            runtime_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

        # Prepare the runtime config so derived output paths and metadata are already resolved.
        prepared_runtime_config = time_distance_module.prepare_runtime_config(runtime_config)
        comparison_runs.append(
            {
                "height_pair": (int(h1_value), int(h2_value)),
                "h1_km": float(shared_heights["height_values_km"][h1_value]),
                "h2_km": float(shared_heights["height_values_km"][h2_value]),
                "runtime_config": prepared_runtime_config,
                "direct_plot_requests": [
                    {
                        "plot_type": "field_strength_comparison",
                        "cube_paths": normalized_paths,
                        "observable": comparison_observable,
                        "h1": int(h1_value),
                        "h2": int(h2_value),
                        "show": False,
                    }
                ],
            }
        )

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "single_cube",
        "cube_paths": normalized_paths,
        "organized_cases": organized_cases,
        "height_values_km": shared_heights["height_values_km"],
        "height_pairs": shared_heights["height_pairs"],
        "comparison_runs": comparison_runs,
    }


def build_single_cube_field_orientation_comparison_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube field-orientation comparison plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Reuse the field-strength plan and then retarget its direct plot requests.
    comparison_plan = build_single_cube_field_strength_comparison_plan(
        base_config,
        time_distance_module,
        cube_paths,
        observable = observable,
        model_atmosphere_path = model_atmosphere_path,
    )

    comparison_plan = copy.deepcopy(comparison_plan)

    # Retarget each comparison request to the orientation-plot mode.
    for comparison_run in comparison_plan["comparison_runs"]:
        comparison_run["direct_plot_requests"][0]["plot_type"] = "field_orientation_comparison"

    # Return the assembled batch plan and its supporting metadata.
    return comparison_plan


def build_single_cube_gaussian_filter_comparison_plan(
    base_config: dict[str, Any],
    time_distance_module: Any,
    cube_paths: Iterable[str | Path],
    filter_params_list: Iterable[dict[str, Any]],
    *,
    observable: str | None = None,
    model_atmosphere_path: str | Path | None = None,
) -> dict[str, Any]:
    '''
    Purpose
    -------
    Build single cube Gaussian-filter comparison plan.
    
    Inputs
    ------
    base_config: dict[str, Any]
        Base configuration used to build the requested result.
    
    time_distance_module: Any
        Time distance module used by this helper.
    
    cube_paths: Iterable[str | Path]
        Cube paths used by this helper.
    
    filter_params_list: Iterable[dict[str, Any]]
        Gaussian-filter parameter sets used by this helper.
    
    observable: str | None
        Observable name used by this helper.
    
    model_atmosphere_path: str | Path | None
        Path to the model-atmosphere file.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Organize the supplied cube paths into the canonical case ordering for this comparison mode.
    organized_cases = organize_single_cube_gaussian_filter_comparison_cases(cube_paths)
    # Work with normalized paths so duplicate or relative inputs collapse to one entry.
    normalized_paths = organized_cases["cube_paths"]
    # Normalize the requested Gaussian filter settings before building any run configs.
    normalized_filter_params = normalize_gaussian_filter_param_list(
        filter_params_list,
        reference_config = base_config,
    )
    # Infer the shared height grid once so every comparison uses the same height pairs.
    shared_heights = generate_shared_single_cube_height_index_pairs(
        time_distance_module,
        normalized_paths,
    )

    # Require at least one shared height pair before building comparison requests.
    if len(shared_heights["height_pairs"]) == 0:
        raise ValueError(
            "Gaussian-filter comparison requires at least two shared heights so h1 < h2 pairs exist."
        )

    # Resolve the optional model-atmosphere path once so every generated config uses the same file.
    resolved_model_atmosphere = (
        str(Path(model_atmosphere_path).expanduser().resolve())
        if model_atmosphere_path not in ["", None]
        else None
    )

    # Resolve the observable once so every run and plot request stays synchronized.
    default_observable = base_config.get("data", {}).get("single_cube", {}).get("observable", "")
    comparison_observable = str(observable) if observable not in ["", None] else str(default_observable)
    # Initialize the run-plan containers that will be filled below.
    run_configs: list[dict[str, Any]] = []
    run_rows: list[dict[str, Any]] = []

    # Step through each normalized Gaussian filter setting.
    for filter_params in normalized_filter_params:
        # Iterate through the ordered simulation cases for the current filter setting.
        for case in organized_cases["ordered_cases"]:
            # Iterate over every valid shared height pair.
            for h1_value, h2_value in shared_heights["height_pairs"]:
                # Work on a deep copy so shared base-config state is not mutated in place.
                runtime_config = copy.deepcopy(base_config)
                runtime_config["data"]["source_type"] = "single_cube"
                runtime_config["data"].setdefault("single_cube", {})
                runtime_config["data"]["single_cube"]["file"] = case["cube_path"]
                runtime_config["data"]["single_cube"]["observable"] = comparison_observable
                runtime_config["data"]["single_cube"]["h1"] = int(h1_value)
                runtime_config["data"]["single_cube"]["h2"] = int(h2_value)

                # Apply the optional model-atmosphere path when one was supplied.
                if resolved_model_atmosphere is not None:
                    runtime_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

                # Apply the selected Gaussian filter settings to this runtime config copy.
                runtime_config = apply_gaussian_filter_params_to_config(runtime_config, filter_params)
                run_configs.append(runtime_config)
                run_rows.append(
                    {
                        "case_label": case["comparison_label"],
                        "cube_path": case["cube_path"],
                        "filter_index": int(filter_params["filter_index"]),
                        "h1": int(h1_value),
                        "h2": int(h2_value),
                        "h1_km": float(shared_heights["height_values_km"][h1_value]),
                        "h2_km": float(shared_heights["height_values_km"][h2_value]),
                    }
                )

    # Build one representative runtime config that td_analysis.ipynb can reuse for plot generation.
    representative_config = copy.deepcopy(base_config)
    representative_config["data"]["source_type"] = "single_cube"
    representative_config["data"].setdefault("single_cube", {})
    representative_config["data"]["single_cube"]["file"] = organized_cases["ordered_cases"][0]["cube_path"]
    representative_config["data"]["single_cube"]["observable"] = comparison_observable
    representative_config["data"]["single_cube"]["h1"] = int(shared_heights["height_pairs"][0][0])
    representative_config["data"]["single_cube"]["h2"] = int(shared_heights["height_pairs"][0][1])

    # Apply the optional model-atmosphere path when one was supplied.
    if resolved_model_atmosphere is not None:
        # Carry the optional atmosphere override into the representative config as well.
        representative_config["data"]["single_cube"]["model_atmosphere_path"] = resolved_model_atmosphere

    # Apply the selected Gaussian filter settings to this runtime config copy.
    representative_config = apply_gaussian_filter_params_to_config(representative_config, normalized_filter_params[0])
    # Prepare the representative runtime config once before attaching the direct plot requests.
    representative_runtime_config = time_distance_module.prepare_runtime_config(representative_config)

    # Accumulate the direct plot requests that td_analysis.ipynb will execute later.
    comparison_runs: list[dict[str, Any]] = []
    # Iterate over every valid shared height pair.
    for h1_value, h2_value in shared_heights["height_pairs"]:
        # Work on a deep copy so shared base-config state is not mutated in place.
        runtime_config = copy.deepcopy(representative_config)
        runtime_config["data"]["single_cube"]["h1"] = int(h1_value)
        runtime_config["data"]["single_cube"]["h2"] = int(h2_value)
        # Prepare the runtime config so derived output paths and metadata are already resolved.
        prepared_runtime_config = time_distance_module.prepare_runtime_config(runtime_config)
        comparison_runs.append(
            {
                "height_pair": (int(h1_value), int(h2_value)),
                "h1_km": float(shared_heights["height_values_km"][h1_value]),
                "h2_km": float(shared_heights["height_values_km"][h2_value]),
                "runtime_config": prepared_runtime_config,
                "direct_plot_requests": [
                    {
                        "plot_type": "gaussian_filter_comparison",
                        "cube_paths": normalized_paths,
                        "filter_params_list": copy.deepcopy(normalized_filter_params),
                        "observable": comparison_observable,
                        "h1": int(h1_value),
                        "h2": int(h2_value),
                        "show": False,
                    }
                ],
            }
        )

    # Return the assembled batch plan and its supporting metadata.
    return {
        "source_type": "single_cube",
        "cube_paths": normalized_paths,
        "organized_cases": organized_cases,
        "filter_params_list": normalized_filter_params,
        "height_values_km": shared_heights["height_values_km"],
        "height_pairs": shared_heights["height_pairs"],
        "run_configs": run_configs,
        "run_rows": run_rows,
        "comparison_runs": comparison_runs,
        "representative_runtime_config": representative_runtime_config,
    }


def describe_runtime_config(runtime_config: dict[str, Any]) -> str:
    '''
    Purpose
    -------
    Describe runtime config.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Read the data block once so the remaining formatting logic stays concise.
    data = runtime_config.get("data", {})
    source_type = str(data.get("source_type", "")).strip().lower()

    # Format paired-cube runs using the two file names.
    if source_type == "paired_cubes":
        # Return the concise human-readable label for this runtime config.
        return f"{Path(data.get('v1', '')).name} vs {Path(data.get('v2', '')).name}"

    # Format single-cube runs with the observable and sampled heights.
    if source_type == "single_cube":
        h1_km = data.get("resolved_h1_km", data.get("h1", "?"))
        h2_km = data.get("resolved_h2_km", data.get("h2", "?"))
        observable = data.get("observable", "")
        # Return the concise human-readable label for this runtime config.
        return f"{Path(data.get('file', '')).name} | {observable} | {h1_km:g} km vs {h2_km:g} km"

    # Return the concise human-readable label for this runtime config.
    return source_type or "unknown run"


def runtime_output_paths(runtime_config: dict[str, Any]) -> dict[str, Path]:
    '''
    Purpose
    -------
    Runtime output paths.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Read the data block once so the remaining formatting logic stays concise.
    data = runtime_config.get("data", {})
    # Collect the standard output products emitted by time_distance.py for this run.
    output_files = {
        "outfile": data.get("outfile", ""),
        "phase_outfile": data.get("phase_outfile", ""),
        "komega_outfile": data.get("komega_outfile", ""),
        "coherence_outfile": data.get("coherence_outfile", ""),
        "orientation_validation_outfile": data.get("orientation_validation_outfile", ""),
    }

    # Return the resolved output paths that belong to this runtime config.
    return {
        key: Path(value).expanduser().resolve()
        for key, value in output_files.items()
        if value not in ["", None]
    }


def batch_outputs_exist(runtime_config: dict[str, Any]) -> bool:
    '''
    Purpose
    -------
    Batch outputs exist.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Resolve the declared output files before deciding whether the run can be reused.
    output_paths = runtime_output_paths(runtime_config)

    # A run with no declared outputs cannot be considered reusable.
    if len(output_paths) == 0:
        # Return the assembled helper result.
        return False

    # Return the assembled helper result.
    return all(path.exists() for path in output_paths.values())


def execute_batch_runs(
    time_distance_module: Any,
    config_file: str | Path,
    run_configs: Iterable[dict[str, Any]],
    *,
    skip_existing: bool = True,
    continue_on_error: bool = False,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    '''
    Purpose
    -------
    Execute batch runs.
    
    Inputs
    ------
    time_distance_module: Any
        Time distance module used by this helper.
    
    config_file: str | Path
        Path to the configuration file.
    
    run_configs: Iterable[dict[str, Any]]
        Run configurations used by this helper.
    
    skip_existing: bool
        Whether existing outputs should be reused.
    
    continue_on_error: bool
        Whether processing should continue after an error.
    
    verbose: bool
        Verbose used by this helper.
    
    Outputs
    -------
    execution_result: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Expand the batch plan into the concrete runtime configs that will be executed.
    run_configs = list(run_configs)
    # Materialize the run list so progress counts and later summaries stay stable.
    run_records: list[dict[str, Any]] = []
    # Record the total number of runs once for progress reporting and result metadata.
    total_runs = len(run_configs)

    # Execute each planned runtime config and collect its batch record.
    for run_index, run_config in enumerate(run_configs, start = 1):
        # Work on a deep copy so shared base-config state is not mutated in place.
        runtime_config = time_distance_module.prepare_runtime_config(copy.deepcopy(run_config))
        # Resolve the configured output files before checking whether the run is reusable.
        output_paths = {
            key: str(path)
            for key, path in runtime_output_paths(runtime_config).items()
        }
        # Initialize the batch record that will capture status, labels, and output paths.
        record = {
            "run_index": int(run_index),
            "total_runs": int(total_runs),
            "status": "",
            "label": describe_runtime_config(runtime_config),
            "source_type": runtime_config["data"]["source_type"],
            "runtime_config": runtime_config,
            "output_paths": output_paths,
        }

        # Print progress only when verbose output is enabled.
        if verbose:
            # Emit a concise progress update for the current batch stage.
            print(f"[{run_index}/{total_runs}] {record['label']}")

        # Reuse completed outputs when the batch is allowed to skip existing files.
        if skip_existing and batch_outputs_exist(runtime_config):
            record["status"] = "skipped_existing"

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print("  skipped existing outputs")

            run_records.append(record)
            continue

        # Run the time-distance pipeline and convert any failure into an explicit batch record.
        try:
            results = time_distance_module.run_time_distance(
                config_file = config_file,
                config_override = copy.deepcopy(run_config),
            )
            record["status"] = "completed"
            record["result_shapes"] = {
                "xc": tuple(results["xc"].shape),
                "phase_diff": tuple(results["phase_diff"].shape),
                "komega": tuple(results["komega"].shape),
                "coherence": tuple(results["coherence"].shape),
            }

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print(f"  wrote {record['output_paths']['outfile']}")
        except Exception as exc:
            record["status"] = "failed"
            record["error"] = f"{exc.__class__.__name__}: {exc}"

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print(f"  failed: {record['error']}")

            run_records.append(record)

            if not continue_on_error:
                raise

            continue

        run_records.append(record)

    # Return the collected execution records for downstream notebook cells.
    return run_records


def build_td_analysis_input_cell_source(
    runtime_config: dict[str, Any],
    *,
    use_config: bool = True,
    direct_plot_requests: list[dict[str, Any]] | list[str] | None = None,
) -> str:
    '''
    Purpose
    -------
    Build time-distance analysis input cell source.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    use_config: bool
        Whether config-driven requests should be used.
    
    direct_plot_requests: list[dict[str, Any]] | list[str] | None
        Direct plot requests supplied by the caller.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Serialize the runtime config and direct requests into readable notebook literals.
    runtime_literal = pprint.pformat(copy.deepcopy(runtime_config), sort_dicts = False, width = 100)
    plot_request_literal = pprint.pformat(
        copy.deepcopy([] if direct_plot_requests is None else direct_plot_requests),
        sort_dicts = False,
        width = 100,
    )

    # Return the notebook cell source that injects the runtime config into td_analysis.ipynb.
    return (
        "# Load the shared configuration file and define the plot requests\n"
        "config_file = resolve_config_file()\n"
        "modules = load_project_modules(config_file)\n"
        f"runtime_config = {runtime_literal}\n"
        "config = prepare_runtime_config(runtime_config)\n\n"
        "# Use a wider k-omega range for simulation cubes\n"
        "if config.get('data', {}).get('source_type', '') == 'single_cube':\n"
        "    PLOT_DEFAULTS['komega_diagram']['vmin'] = -80.0\n\n"
        "# Comparison-plot execution defaults\n"
        "comparison_execution_mode = 'load'\n"
        "comparison_missing_data_behavior = 'error'\n\n"
        f"use_config = {bool(use_config)}\n"
        f"direct_plot_requests = {plot_request_literal}\n\n"
        "print(config_file)\n"
        "print(config['data']['outfile'])\n"
    )


def import_td_analysis_notebook_dependencies():
    '''
    Purpose
    -------
    Import time-distance analysis notebook dependencies.
    
    Inputs
    ------
    None
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Import the notebook execution packages lazily so the rest of the batch helpers stay usable.
    try:
        import nbformat
        from nbclient import NotebookClient
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Executing td_analysis.ipynb requires the 'nbformat' and 'nbclient' packages."
        ) from exc

    # Return the imported notebook-execution helpers.
    return nbformat, NotebookClient


def execute_td_analysis_notebook(
    runtime_config: dict[str, Any],
    *,
    analysis_notebook: str | Path | None = None,
    output_dir: str | Path | None = None,
    use_config: bool = True,
    direct_plot_requests: list[dict[str, Any]] | list[str] | None = None,
    timeout: int = 3600,
    kernel_name: str | None = None,
) -> Path:
    '''
    Purpose
    -------
    Execute time-distance analysis notebook.
    
    Inputs
    ------
    runtime_config: dict[str, Any]
        Runtime config used by this helper.
    
    analysis_notebook: str | Path | None
        Path to the analysis notebook that will be executed.
    
    output_dir: str | Path | None
        Output directory used by this helper.
    
    use_config: bool
        Whether config-driven requests should be used.
    
    direct_plot_requests: list[dict[str, Any]] | list[str] | None
        Direct plot requests supplied by the caller.
    
    timeout: int
        Execution timeout in seconds.
    
    kernel_name: str | None
        Kernel name used by this helper.
    
    Outputs
    -------
    execution_result: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Load the notebook execution stack only when analysis execution is requested.
    nbformat, NotebookClient = import_td_analysis_notebook_dependencies()

    if analysis_notebook in ["", None]:
        # Resolve the shared batch workspace before deriving any runtime paths.
        analysis_notebook_path = (resolve_td_batch_module_dir() / "td_analysis.ipynb").resolve()
    else:
        # Resolve the source analysis notebook path, defaulting to the local project copy.
        analysis_notebook_path = Path(analysis_notebook).expanduser().resolve()

    # Load the source analysis notebook from disk before rewriting its input cell.
    with analysis_notebook_path.open("r", encoding = "utf-8") as notebook_handle:
        notebook = nbformat.read(notebook_handle, as_version = 4)

    target_cell_index = None
    # Find the input-cell marker so the runtime config can be injected into the notebook.
    for index, cell in enumerate(notebook.cells):
        if cell.get("cell_type") == "code" and ANALYSIS_INPUT_CELL_MARKER in cell.get("source", ""):
            target_cell_index = index
            break

    # Fail loudly when the marker cell cannot be found in the analysis notebook.
    if target_cell_index is None:
        raise ValueError(
            f"Could not find the td_analysis input cell marker in {analysis_notebook_path}."
        )

    # Inject the runtime-specific input cell so the executed notebook uses this batch config.
    notebook.cells[target_cell_index]["source"] = build_td_analysis_input_cell_source(
        runtime_config,
        use_config = use_config,
        direct_plot_requests = direct_plot_requests,
    )

    # Default executed notebooks to an analysis_notebooks directory beside the data outputs.
    if output_dir in ["", None]:
        output_directory = (
            Path(runtime_config["paths"]["data_output_dir"]).expanduser().resolve()
            / "analysis_notebooks"
        )
    else:
        output_directory = Path(output_dir).expanduser().resolve()

    output_directory.mkdir(parents = True, exist_ok = True)
    # Build the output notebook path from the runtime config so each analysis run is traceable.
    executed_notebook = output_directory / f"{Path(runtime_config['data']['komega_outfile']).stem}_td_analysis.ipynb"

    # Configure the notebook client to run from the analysis notebook directory.
    client_kwargs = {
        "timeout": int(timeout),
        "resources": {"metadata": {"path": str(analysis_notebook_path.parent)}},
    }

    # Honor an explicit kernel override when one was supplied.
    if kernel_name not in ["", None]:
        # Add the explicit kernel name only when the caller requested one.
        client_kwargs["kernel_name"] = str(kernel_name)

    # Create the notebook client that will execute the modified analysis notebook.
    client = NotebookClient(notebook, **client_kwargs)
    client.execute()

    # Persist the executed notebook to disk after the client finishes running it.
    with executed_notebook.open("w", encoding = "utf-8") as notebook_handle:
        nbformat.write(notebook, notebook_handle)

    # Return the path to the executed analysis notebook.
    return executed_notebook


def execute_td_analysis_for_batch(
    run_records: Iterable[dict[str, Any]],
    *,
    analysis_notebook: str | Path | None = None,
    output_dir: str | Path | None = None,
    use_config: bool = True,
    direct_plot_requests: list[dict[str, Any]] | list[str] | None = None,
    timeout: int = 3600,
    kernel_name: str | None = None,
    continue_on_error: bool = False,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    '''
    Purpose
    -------
    Execute time-distance analysis for batch.
    
    Inputs
    ------
    run_records: Iterable[dict[str, Any]]
        Run records used by this helper.
    
    analysis_notebook: str | Path | None
        Path to the analysis notebook that will be executed.
    
    output_dir: str | Path | None
        Output directory used by this helper.
    
    use_config: bool
        Whether config-driven requests should be used.
    
    direct_plot_requests: list[dict[str, Any]] | list[str] | None
        Direct plot requests supplied by the caller.
    
    timeout: int
        Execution timeout in seconds.
    
    kernel_name: str | None
        Kernel name used by this helper.
    
    continue_on_error: bool
        Whether processing should continue after an error.
    
    verbose: bool
        Verbose used by this helper.
    
    Outputs
    -------
    execution_result: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Materialize the run list so progress counts and later summaries stay stable.
    run_records = list(run_records)
    # Initialize the analysis-record list that will capture notebook execution outcomes.
    analysis_records: list[dict[str, Any]] = []
    # Keep only the runs that completed or were safely skipped with existing outputs.
    runnable_records = [
        record for record in run_records if record.get("status", "") in ["completed", "skipped_existing"]
    ]
    # Track any missing notebook-execution dependency so it can be reported uniformly.
    missing_dependency_error = None

    # Run the analysis notebook for this batch entry and preserve its outcome in the record list.
    try:
        import_td_analysis_notebook_dependencies()
    except ModuleNotFoundError as exc:
        # Track any missing notebook-execution dependency so it can be reported uniformly.
        missing_dependency_error = f"{exc.__class__.__name__}: {exc}"

    # Return explicit skipped records when notebook-execution dependencies are unavailable.
    if missing_dependency_error is not None:
        # Print progress only when verbose output is enabled.
        if verbose:
            # Emit a concise progress update for the current batch stage.
            print(f"Analysis notebook execution unavailable: {missing_dependency_error}")

        # Execute the analysis notebook for each runnable batch record.
        for analysis_index, record in enumerate(runnable_records, start = 1):
            analysis_records.append(
                {
                    "analysis_index": int(analysis_index),
                    "total_runs": int(len(runnable_records)),
                    "label": record["label"],
                    "source_type": record["source_type"],
                    "runtime_config": record["runtime_config"],
                    "status": "skipped_missing_dependency",
                    "error": missing_dependency_error,
                }
            )

        # Tally the run statuses across the completed batch records.
        for record in run_records:
            # Add placeholder analysis records for parent runs that never completed.
            if record.get("status", "") == "failed":
                analysis_records.append(
                    {
                        "analysis_index": None,
                        "total_runs": int(len(runnable_records)),
                        "label": record["label"],
                        "source_type": record["source_type"],
                        "runtime_config": record["runtime_config"],
                        "status": "skipped_parent_failure",
                    }
                )

        # Return the collected execution records for downstream notebook cells.
        return analysis_records

    # Execute the analysis notebook for each runnable batch record.
    for analysis_index, record in enumerate(runnable_records, start = 1):
        # Initialize the per-run analysis record before executing the notebook.
        analysis_record = {
            "analysis_index": int(analysis_index),
            "total_runs": int(len(runnable_records)),
            "label": record["label"],
            "source_type": record["source_type"],
            "runtime_config": record["runtime_config"],
            "status": "",
        }

        # Print progress only when verbose output is enabled.
        if verbose:
            # Emit a concise progress update for the current batch stage.
            print(f"[analysis {analysis_index}/{len(runnable_records)}] {record['label']}")

        # Run the analysis notebook for this batch entry and preserve its outcome in the record list.
        try:
            # Build the output notebook path from the runtime config so each analysis run is traceable.
            executed_notebook = execute_td_analysis_notebook(
                record["runtime_config"],
                analysis_notebook = analysis_notebook,
                output_dir = output_dir,
                use_config = use_config,
                direct_plot_requests = direct_plot_requests,
                timeout = timeout,
                kernel_name = kernel_name,
            )
            # Initialize the per-run analysis record before executing the notebook.
            analysis_record["status"] = "completed"
            analysis_record["executed_notebook"] = str(executed_notebook)

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print(f"  executed {executed_notebook}")
        except Exception as exc:
            # Initialize the per-run analysis record before executing the notebook.
            analysis_record["status"] = "failed"
            analysis_record["error"] = f"{exc.__class__.__name__}: {exc}"

            # Print progress only when verbose output is enabled.
            if verbose:
                # Emit a concise progress update for the current batch stage.
                print(f"  failed: {analysis_record['error']}")

            analysis_records.append(analysis_record)

            if not continue_on_error:
                raise

            continue

        analysis_records.append(analysis_record)

    # Tally the run statuses across the completed batch records.
    for record in run_records:
        # Add placeholder analysis records for parent runs that never completed.
        if record.get("status", "") == "failed":
            analysis_records.append(
                {
                    "analysis_index": None,
                    "total_runs": int(len(runnable_records)),
                    "label": record["label"],
                    "source_type": record["source_type"],
                    "runtime_config": record["runtime_config"],
                    "status": "skipped_parent_failure",
                }
            )

    # Return the collected execution records for downstream notebook cells.
    return analysis_records


def summarize_batch_records(
    run_records: Iterable[dict[str, Any]],
    analysis_records: Iterable[dict[str, Any]] | None = None,
) -> dict[str, dict[str, int]]:
    '''
    Purpose
    -------
    Summarize batch records.
    
    Inputs
    ------
    run_records: Iterable[dict[str, Any]]
        Run records used by this helper.
    
    analysis_records: Iterable[dict[str, Any]] | None
        Analysis records used by this helper.
    
    Outputs
    -------
    summary: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Initialize separate counters for the batch and analysis stages.
    summary = {
        "runs": {"completed": 0, "skipped_existing": 0, "failed": 0},
        "analysis": {
            "completed": 0,
            "failed": 0,
            "skipped_missing_dependency": 0,
            "skipped_parent_failure": 0,
        },
    }

    # Tally the run statuses across the completed batch records.
    for record in run_records:
        status = record.get("status", "")
        # Only tally statuses that belong to the run-summary table.
        if status in summary["runs"]:
            summary["runs"][status] += 1

    # Tally the analysis statuses across the executed notebook records.
    for record in [] if analysis_records is None else analysis_records:
        status = record.get("status", "")
        # Only tally statuses that belong to the analysis-summary table.
        if status in summary["analysis"]:
            summary["analysis"][status] += 1

    # Emit a concise progress update for the current batch stage.
    print(
        "Runs:"
        f" completed={summary['runs']['completed']},"
        f" skipped_existing={summary['runs']['skipped_existing']},"
        f" failed={summary['runs']['failed']}"
    )

    # Print the analysis summary only when analysis records were supplied.
    if analysis_records is not None:
        # Emit a concise progress update for the current batch stage.
        print(
            "Analysis:"
            f" completed={summary['analysis']['completed']},"
            f" failed={summary['analysis']['failed']},"
            f" skipped_missing_dependency={summary['analysis']['skipped_missing_dependency']},"
            f" skipped_parent_failure={summary['analysis']['skipped_parent_failure']}"
        )

    # Return the compact status summary for the batch and analysis stages.
    return summary


CONFIG_FILE = None  # Set this to a config.py path to override automatic resolution.
ANALYSIS_NOTEBOOK = None  # Set this to a td_analysis.ipynb path to override automatic resolution.


In [ ]:
# Paired Cubes Batch Cell
config_file, td, base_config = load_mode_base_config(
    "paired_cubes",
    config_file = CONFIG_FILE,
)

# Optional shared overrides before the batch is built.
# base_config["paths"]["data_output_dir"] = "/path/to/time_distance_outputs"
# base_config["paths"]["figure_dir"] = "/path/to/figures"

cube_paths = [
    base_config["data"]["paired_cubes"]["v1"],
    base_config["data"]["paired_cubes"]["v2"],
    # "/path/to/another_cube.fits",
]

delta_z_km = base_config["data"]["paired_cubes"]["delta_z_km"]
p_dx_Mm = base_config["data"]["paired_cubes"]["p_dx_Mm"]
dt = base_config["data"]["paired_cubes"]["dt"]

# Set the batch-run behavior for execution and follow-up analysis.
skip_existing = True
continue_on_error = True
run_analysis = True
analysis_timeout = 3600

# Build the list of unique paired-cube runs from the shared settings.
paired_plan = build_paired_cubes_batch_plan(
    base_config,
    td,
    cube_paths,
    delta_z_km = delta_z_km,
    p_dx_Mm = p_dx_Mm,
    dt = dt,
)

v1_list = paired_plan["v1_list"]
v2_list = paired_plan["v2_list"]

# Preview the planned file pairs before running the pipeline.
print(f"Queued {len(paired_plan['run_configs'])} unique paired-cube runs.")
for index, (v1, v2) in enumerate(paired_plan["file_pairs"], start = 1):
    print(f"[{index:03d}] {Path(v1).name} <> {Path(v2).name}")

# Execute the planned runs and print the saved outputs.
paired_run_records = execute_batch_runs(
    td,
    config_file,
    paired_plan["run_configs"],
    skip_existing = skip_existing,
    continue_on_error = continue_on_error,
)

# Optionally execute the analysis notebook for each completed run.
if run_analysis:
    paired_analysis_records = execute_td_analysis_for_batch(
        paired_run_records,
        analysis_notebook = ANALYSIS_NOTEBOOK,
        timeout = analysis_timeout,
        continue_on_error = continue_on_error,
    )
else:
    paired_analysis_records = []

# Summarize the batch outcome for quick inspection in the notebook.
paired_summary = summarize_batch_records(
    paired_run_records,
    paired_analysis_records,
)

paired_summary


In [ ]:
# Single Cube Batch Cell
config_file, td, base_config = load_mode_base_config(
    "single_cube",
    config_file = CONFIG_FILE,
)

# Optional shared overrides before the batch is built.
# base_config["paths"]["data_output_dir"] = "/path/to/time_distance_outputs"
# base_config["paths"]["figure_dir"] = "/path/to/figures"

cube_paths = [
    base_config["data"]["single_cube"]["file"],
    # "/path/to/another_cube.nc",
]

observable = base_config["data"]["single_cube"]["observable"]
model_atmosphere_path = base_config["data"]["single_cube"].get("model_atmosphere_path", "")

# Set the batch-run behavior for execution and follow-up analysis.
skip_existing = True
continue_on_error = True
run_analysis = True
analysis_timeout = 3600

# Build the list of single-cube runs from the shared settings.
single_plan = build_single_cube_batch_plan(
    base_config,
    td,
    cube_paths,
    observable = observable,
    model_atmosphere_path = model_atmosphere_path,
)

height_pairs_by_cube = single_plan["height_pairs_by_cube"]
height_pair_rows = single_plan["height_pair_rows"]

# Preview the planned height-pair coverage before running the pipeline.
print(f"Queued {len(single_plan['run_configs'])} single-cube runs.")
for cube_path, height_pairs in height_pairs_by_cube.items():
    print(f"{Path(cube_path).name}: {len(height_pairs)} height pairs")

# Execute the planned runs and print the saved outputs.
single_run_records = execute_batch_runs(
    td,
    config_file,
    single_plan["run_configs"],
    skip_existing = skip_existing,
    continue_on_error = continue_on_error,
)

# Optionally execute the analysis notebook for each completed run.
if run_analysis:
    single_analysis_records = execute_td_analysis_for_batch(
        single_run_records,
        analysis_notebook = ANALYSIS_NOTEBOOK,
        timeout = analysis_timeout,
        continue_on_error = continue_on_error,
    )
else:
    single_analysis_records = []

# Summarize the batch outcome for quick inspection in the notebook.
single_summary = summarize_batch_records(
    single_run_records,
    single_analysis_records,
)

single_summary


In [ ]:
# Single Cube Field-Strength Comparison Analysis Cell
config_file, td, base_config = load_mode_base_config(
    "single_cube",
    config_file = CONFIG_FILE,
)

# Optional shared overrides before the comparison batch is built.
# base_config["paths"]["data_output_dir"] = "/path/to/time_distance_outputs"
# base_config["paths"]["figure_dir"] = "/path/to/figures"

cube_paths = [
    base_config["data"]["single_cube"]["file"],
    # "/path/to/another_cube.nc",
]

observable = base_config["data"]["single_cube"]["observable"]
model_atmosphere_path = base_config["data"]["single_cube"].get("model_atmosphere_path", "")

# Set the execution behavior for the parent runs and comparison analysis.
skip_existing = True
continue_on_error = True
run_time_distance_batch = True
comparison_run_analysis = True
comparison_timeout = 3600

# Optionally regenerate the parent single-cube runs before plotting comparisons.
if run_time_distance_batch:
    comparison_single_plan = build_single_cube_batch_plan(
        base_config,
        td,
        cube_paths,
        observable = observable,
        model_atmosphere_path = model_atmosphere_path,
    )
        # Execute the planned runs and print the saved outputs.
    comparison_parent_run_records = execute_batch_runs(
        td,
        config_file,
        comparison_single_plan["run_configs"],
        skip_existing = skip_existing,
        continue_on_error = continue_on_error,
    )
else:
    comparison_parent_run_records = []

# Build the comparison requests from the configured cube set.
comparison_plan = build_single_cube_field_strength_comparison_plan(
    base_config,
    td,
    cube_paths,
    observable = observable,
    model_atmosphere_path = model_atmosphere_path,
)

# Preview the requested field-strength comparisons before execution.
print(f"Queued {len(comparison_plan['comparison_runs'])} field-strength comparison plots.")
for index, comparison_run in enumerate(comparison_plan['comparison_runs'], start = 1):
    print(
        f"[{index:03d}] "
        f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
    )

# Collect the notebook-execution status for each comparison request.
comparison_analysis_records = []

# Execute the analysis notebook for each comparison request when enabled.
if comparison_run_analysis:
    total_comparisons = len(comparison_plan['comparison_runs'])
    for comparison_index, comparison_run in enumerate(comparison_plan['comparison_runs'], start = 1):
        label = f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
        print(f"[comparison {comparison_index}/{total_comparisons}] {label}")

        try:
            executed_notebook = execute_td_analysis_notebook(
                comparison_run['runtime_config'],
                analysis_notebook = ANALYSIS_NOTEBOOK,
                use_config = False,
                direct_plot_requests = comparison_run['direct_plot_requests'],
                timeout = comparison_timeout,
            )
            comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'completed',
                    'executed_notebook': str(executed_notebook),
                }
            )
            print(f"  wrote {executed_notebook}")
        except Exception as exc:
            error_message = f"{exc.__class__.__name__}: {exc}"
            comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'failed',
                    'error': error_message,
                }
            )
            print(f"  failed: {error_message}")
            if not continue_on_error:
                raise

comparison_analysis_records


In [ ]:
# Single Cube Field-Orientation Comparison Analysis Cell
config_file, td, orientation_base_config = load_mode_base_config(
    "single_cube",
    config_file = CONFIG_FILE,
)

# Optional shared overrides before the comparison batch is built.
# orientation_base_config["paths"]["data_output_dir"] = "/path/to/time_distance_outputs"
# orientation_base_config["paths"]["figure_dir"] = "/path/to/figures"

orientation_cube_paths = [
    orientation_base_config["data"]["single_cube"]["file"],
    # "/path/to/another_cube.nc",
]

orientation_observable = orientation_base_config["data"]["single_cube"]["observable"]
orientation_model_atmosphere_path = orientation_base_config["data"]["single_cube"].get("model_atmosphere_path", "")

# Set the execution behavior for the parent runs and comparison analysis.
orientation_skip_existing = True
orientation_continue_on_error = True
orientation_run_time_distance_batch = True
orientation_comparison_run_analysis = True
orientation_comparison_timeout = 3600

# Optionally regenerate the parent single-cube runs before plotting comparisons.
if orientation_run_time_distance_batch:
    orientation_single_plan = build_single_cube_batch_plan(
        orientation_base_config,
        td,
        orientation_cube_paths,
        observable = orientation_observable,
        model_atmosphere_path = orientation_model_atmosphere_path,
    )
        # Execute the planned runs and print the saved outputs.
    orientation_parent_run_records = execute_batch_runs(
        td,
        config_file,
        orientation_single_plan["run_configs"],
        skip_existing = orientation_skip_existing,
        continue_on_error = orientation_continue_on_error,
    )
else:
    orientation_parent_run_records = []

# Build the comparison requests from the configured cube set.
orientation_comparison_plan = build_single_cube_field_orientation_comparison_plan(
    orientation_base_config,
    td,
    orientation_cube_paths,
    observable = orientation_observable,
    model_atmosphere_path = orientation_model_atmosphere_path,
)

# Preview the requested field-orientation comparisons before execution.
print(f"Queued {len(orientation_comparison_plan['comparison_runs'])} field-orientation comparison plots.")
for index, comparison_run in enumerate(orientation_comparison_plan['comparison_runs'], start = 1):
    print(
        f"[{index:03d}] "
        f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
    )

# Collect the notebook-execution status for each comparison request.
orientation_comparison_analysis_records = []

# Execute the analysis notebook for each comparison request when enabled.
if orientation_comparison_run_analysis:
    total_comparisons = len(orientation_comparison_plan['comparison_runs'])
    for comparison_index, comparison_run in enumerate(orientation_comparison_plan['comparison_runs'], start = 1):
        label = f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
        print(f"[comparison {comparison_index}/{total_comparisons}] {label}")

        try:
            executed_notebook = execute_td_analysis_notebook(
                comparison_run['runtime_config'],
                analysis_notebook = ANALYSIS_NOTEBOOK,
                use_config = False,
                direct_plot_requests = comparison_run['direct_plot_requests'],
                timeout = orientation_comparison_timeout,
            )
            orientation_comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'completed',
                    'executed_notebook': str(executed_notebook),
                }
            )
            print(f"  wrote {executed_notebook}")
        except Exception as exc:
            error_message = f"{exc.__class__.__name__}: {exc}"
            orientation_comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'failed',
                    'error': error_message,
                }
            )
            print(f"  failed: {error_message}")
            if not orientation_continue_on_error:
                raise

orientation_comparison_analysis_records


In [ ]:
# Single Cube Gaussian-Filter Comparison Analysis Cell
config_file, td, gaussian_comparison_base_config = load_mode_base_config(
    "single_cube",
    config_file = CONFIG_FILE,
)

# Optional shared overrides before the comparison batch is built.
# gaussian_comparison_base_config["paths"]["data_output_dir"] = "/path/to/time_distance_outputs"
# gaussian_comparison_base_config["paths"]["figure_dir"] = "/path/to/figures"

gaussian_comparison_cube_paths = [
    gaussian_comparison_base_config["data"]["single_cube"]["file"],
    # "/path/to/z0_0G_cube.nc",
    # "/path/to/hx_10G_cube.nc",
    # "/path/to/hx_50G_cube.nc",
    # "/path/to/hx_100G_cube.nc",
    # "/path/to/vx_10G_cube.nc",
    # "/path/to/vx_50G_cube.nc",
    # "/path/to/vx_100G_cube.nc",
]

gaussian_base_params = dict(gaussian_comparison_base_config.get("filtering", {}).get("gaussian", {}))
gaussian_comparison_filter_params_list = [
    dict(gaussian_base_params),
    # {**gaussian_base_params, "central_k": 3.0, "width_k": 1.0, "central_f": 2.0, "width_f": 1.0},
]

gaussian_comparison_observable = gaussian_comparison_base_config["data"]["single_cube"]["observable"]
gaussian_comparison_model_atmosphere_path = gaussian_comparison_base_config["data"]["single_cube"].get("model_atmosphere_path", "")

# Set the execution behavior for the parent runs and Gaussian-filter comparisons.
gaussian_comparison_skip_existing = True
gaussian_comparison_continue_on_error = True
gaussian_comparison_run_time_distance_batch = True
gaussian_comparison_run_analysis = True
gaussian_comparison_timeout = 5400

# Build the comparison requests from the configured cube set and filter parameters.
gaussian_comparison_plan = build_single_cube_gaussian_filter_comparison_plan(
    gaussian_comparison_base_config,
    td,
    gaussian_comparison_cube_paths,
    gaussian_comparison_filter_params_list,
    observable = gaussian_comparison_observable,
    model_atmosphere_path = gaussian_comparison_model_atmosphere_path,
)

# Preview the requested Gaussian-filter comparisons before execution.
print(
    f"Queued {len(gaussian_comparison_plan['comparison_runs'])} Gaussian-filter comparison plots "
    f"from {len(gaussian_comparison_plan['run_configs'])} single_cube runs."
)
for index, comparison_run in enumerate(gaussian_comparison_plan['comparison_runs'], start = 1):
    print(
        f"[{index:03d}] "
        f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
    )

# Optionally regenerate the parent single-cube runs before plotting comparisons.
if gaussian_comparison_run_time_distance_batch:
    # Execute the planned runs and print the saved outputs.
    gaussian_comparison_parent_run_records = execute_batch_runs(
        td,
        config_file,
        gaussian_comparison_plan["run_configs"],
        skip_existing = gaussian_comparison_skip_existing,
        continue_on_error = gaussian_comparison_continue_on_error,
    )
else:
    gaussian_comparison_parent_run_records = []

# Collect the notebook-execution status for each comparison request.
gaussian_comparison_analysis_records = []

# Execute the analysis notebook for each comparison request when enabled.
if gaussian_comparison_run_analysis:
    total_comparisons = len(gaussian_comparison_plan['comparison_runs'])
    for comparison_index, comparison_run in enumerate(gaussian_comparison_plan['comparison_runs'], start = 1):
        label = f"{comparison_run['h1_km']:g} km vs {comparison_run['h2_km']:g} km"
        print(f"[comparison {comparison_index}/{total_comparisons}] {label}")

        try:
            executed_notebook = execute_td_analysis_notebook(
                comparison_run['runtime_config'],
                analysis_notebook = ANALYSIS_NOTEBOOK,
                use_config = False,
                direct_plot_requests = comparison_run['direct_plot_requests'],
                timeout = gaussian_comparison_timeout,
            )
            gaussian_comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'completed',
                    'executed_notebook': str(executed_notebook),
                }
            )
            print(f"  wrote {executed_notebook}")
        except Exception as exc:
            error_message = f"{exc.__class__.__name__}: {exc}"
            gaussian_comparison_analysis_records.append(
                {
                    'comparison_index': int(comparison_index),
                    'total_comparisons': int(total_comparisons),
                    'label': label,
                    'status': 'failed',
                    'error': error_message,
                }
            )
            print(f"  failed: {error_message}")
            if not gaussian_comparison_continue_on_error:
                raise

gaussian_comparison_analysis_records
